In [0]:

df = spark.read.csv(
    "/Volumes/workspace/default/loan_project/application_train.csv",
    header=True,
    inferSchema=True
)

print("Rows:", df.count())
print("Columns:", len(df.columns))
df.printSchema()

Rows: 307511
Columns: 122
root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- TARGET: integer (nullable = true)
 |-- NAME_CONTRACT_TYPE: string (nullable = true)
 |-- CODE_GENDER: string (nullable = true)
 |-- FLAG_OWN_CAR: string (nullable = true)
 |-- FLAG_OWN_REALTY: string (nullable = true)
 |-- CNT_CHILDREN: integer (nullable = true)
 |-- AMT_INCOME_TOTAL: double (nullable = true)
 |-- AMT_CREDIT: double (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)
 |-- AMT_GOODS_PRICE: double (nullable = true)
 |-- NAME_TYPE_SUITE: string (nullable = true)
 |-- NAME_INCOME_TYPE: string (nullable = true)
 |-- NAME_EDUCATION_TYPE: string (nullable = true)
 |-- NAME_FAMILY_STATUS: string (nullable = true)
 |-- NAME_HOUSING_TYPE: string (nullable = true)
 |-- REGION_POPULATION_RELATIVE: double (nullable = true)
 |-- DAYS_BIRTH: integer (nullable = true)
 |-- DAYS_EMPLOYED: integer (nullable = true)
 |-- DAYS_REGISTRATION: double (nullable = true)
 |-- DAYS_ID_PUBLISH: integer (nullab

In [0]:
# ============================================================
# 01_BRONZE_INGESTION
# Load raw CSV → Bronze Delta Table (no transformations)
# ============================================================

RAW_PATH = "/Volumes/workspace/default/loan_project/application_train.csv"
BRONZE_TABLE = "workspace.default.bronze_loan_applications"

# Read raw file
df_raw = spark.read.csv(RAW_PATH, header=True, inferSchema=True)

print("✅ Rows:", df_raw.count())
print("✅ Columns:", len(df_raw.columns))

✅ Rows: 307511
✅ Columns: 122


In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Add ingestion metadata
df_bronze = df_raw \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", lit("application_train.csv")) \
    .withColumn("_layer", lit("bronze"))

print("✅ Metadata columns added")
df_bronze.printSchema()

✅ Metadata columns added
root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- TARGET: integer (nullable = true)
 |-- NAME_CONTRACT_TYPE: string (nullable = true)
 |-- CODE_GENDER: string (nullable = true)
 |-- FLAG_OWN_CAR: string (nullable = true)
 |-- FLAG_OWN_REALTY: string (nullable = true)
 |-- CNT_CHILDREN: integer (nullable = true)
 |-- AMT_INCOME_TOTAL: double (nullable = true)
 |-- AMT_CREDIT: double (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)
 |-- AMT_GOODS_PRICE: double (nullable = true)
 |-- NAME_TYPE_SUITE: string (nullable = true)
 |-- NAME_INCOME_TYPE: string (nullable = true)
 |-- NAME_EDUCATION_TYPE: string (nullable = true)
 |-- NAME_FAMILY_STATUS: string (nullable = true)
 |-- NAME_HOUSING_TYPE: string (nullable = true)
 |-- REGION_POPULATION_RELATIVE: double (nullable = true)
 |-- DAYS_BIRTH: integer (nullable = true)
 |-- DAYS_EMPLOYED: integer (nullable = true)
 |-- DAYS_REGISTRATION: double (nullable = true)
 |-- DAYS_ID_PUBLISH: integer (nullabl

In [0]:
# Find your actual catalog and schema
spark.sql("SELECT current_catalog(), current_schema()").show()


+-----------------+----------------+
|current_catalog()|current_schema()|
+-----------------+----------------+
|        workspace|         default|
+-----------------+----------------+



In [0]:
# Write as Delta table (overwrite for idempotency)
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(BRONZE_TABLE)

print(f"✅ Bronze table written: {BRONZE_TABLE}")

✅ Bronze table written: workspace.default.bronze_loan_applications


In [0]:
# Verify the table
df_check = spark.table(BRONZE_TABLE)
print("✅ Bronze table row count:", df_check.count())
print("✅ Columns:", len(df_check.columns))

# Show sample
df_check.select("SK_ID_CURR", "TARGET", "NAME_CONTRACT_TYPE", "AMT_CREDIT", "_ingested_at").show(5)

✅ Bronze table row count: 307511
✅ Columns: 125
+----------+------+------------------+----------+--------------------+
|SK_ID_CURR|TARGET|NAME_CONTRACT_TYPE|AMT_CREDIT|        _ingested_at|
+----------+------+------------------+----------+--------------------+
|    387559|     0|        Cash loans|  896602.5|2026-06-15 13:46:...|
|    387560|     0|        Cash loans|  225000.0|2026-06-15 13:46:...|
|    387561|     0|        Cash loans|  677664.0|2026-06-15 13:46:...|
|    387562|     0|        Cash loans|  384048.0|2026-06-15 13:46:...|
|    387563|     0|        Cash loans| 1113133.5|2026-06-15 13:46:...|
+----------+------+------------------+----------+--------------------+
only showing top 5 rows


In [0]:
# Show Delta history — proves Delta Lake is working
spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}").show(5, truncate=False)

+-------+-------------------+--------------+-----------------------+---------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+-----------------------+-----------+-----------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------------+------------+--------------------------------------------------+
|version|timestamp          |userId        |userName               |operation                        |operationParameters                                                                                          